In [ ]:
import torch.nn as nn
from torchvision.models.vision_transformer import vit_b_16

In [ ]:
def create_custom_vit(num_classes=2, pretrained=True):
    """
    Initializes a ViT model pre-trained on ImageNet and replaces its classifier head
    for binary classification.

    Args:
        num_classes (int): Number of output classes. Default is 2.
        pretrained (bool): Whether to load the pre-trained weights. Default is True.
        device (torch.device): The device to move the model to.

    Returns:
        model (nn.Module): The modified ViT model.
    """
    model = vit_b_16(pretrained=pretrained)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    model = model.to(mps_device)
    return model

In [ ]:
# Store metrics from each fold
fold_metrics = {}

for fold, (train_idx, val_idx) in enumerate(kf.split(exp_ds)):
    print(f"\nStarting Fold {fold+1}")

    # Create subsets for the current fold
    train_subset = Subset(exp_ds, train_idx)
    val_subset = Subset(exp_ds, val_idx)
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    
    # Initialize the model for this fold
    model = vit_b_16(pretrained=True)
    # Modify the classifier head to output 2 classes (binary classification)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, 2)
    model.to(mps_device)  # move model to MPS device (or CPU/GPU as available)
    
    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training loop for the current fold
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(mps_device), labels.to(mps_device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        train_loss = running_loss / total
        train_acc = correct / total
        
        # Evaluation on validation set
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(mps_device), labels.to(mps_device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)
        
        val_loss /= total_val
        val_acc = correct_val / total_val
        
        print(f"Fold {fold+1} | Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # Store the validation accuracy for this fold
    fold_metrics[fold] = val_acc

# After all folds, you can compute average performance
avg_val_acc = sum(fold_metrics.values()) / len(fold_metrics)
print(f"\nAverage Validation Accuracy over {kf.get_n_splits()} folds: {avg_val_acc:.4f}")